In [104]:
# imports
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from functools import reduce
from sklearn.linear_model import Lasso, LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, 
    mean_squared_error, 
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    classification_report, 
    roc_curve, 
    roc_auc_score
)
from sklearn.utils import resample
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [105]:
# load all coefficient dfs
coefficient_df_2018 = pd.read_csv('coefficient_df_2018_all.csv')
coefficient_df_2019 = pd.read_csv('coefficient_df_2019_all.csv')
coefficient_df_2020 = pd.read_csv('coefficient_df_2020_all.csv')
coefficient_df_2021 = pd.read_csv('coefficient_df_2021_all.csv')
coefficient_df_2022 = pd.read_csv('coefficient_df_2022_all.csv')

In [106]:
# function that deletes all of the zero coefficients and sorts it
def deletezero_sort(df):
    df = df[df['Coefficient'] != 0]
    df['abs_coefficient'] = df['Coefficient'].abs()
    df['odds_ratio'] = np.exp(df['Coefficient'])
    df = df.sort_values(by='abs_coefficient', ascending=False)
    return df

In [107]:
# function that only keeps race, sex, and ethnicity features and sorts it
def filter_features_by_prefix(df):
    prefixes = ['derived_ethnicity', 'derived_race', 'applicant_ethnicity', 'co-applicant_ethnicity', 'applicant_race', 'co-applicant_race']
    filtered_coefficients = pd.DataFrame()
    for prefix in prefixes:
        filtered_coefficients = pd.concat([filtered_coefficients, df[df['Feature'].str.startswith(prefix)]])
    filtered_coefficients['abs_coefficient'] = filtered_coefficients['Coefficient'].abs()
    filtered_coefficients['odds_ratio'] = np.exp(filtered_coefficients['Coefficient'])
    filtered_coefficients = filtered_coefficients.sort_values(by='abs_coefficient', ascending=False)
    return filtered_coefficients

## 2018

In [108]:
coefficient_sorted_2018 = deletezero_sort(coefficient_df_2018)
coefficient_sorted_2018

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_91328/2794036433.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['abs_coefficient'] = df['Coefficient'].abs()
/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_91328/2794036433.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['odds_ratio'] = np.exp(df['Coefficient'])


,Feature,Coefficient,abs_coefficient,odds_ratio
0,const,4.327318,4.327318,75.740905
40,purchaser_type_Not applicable,-2.097301,2.097301,0.122787
51,hoepa_status_Not applicable,-1.494363,1.494363,0.224391
50,hoepa_status_Not a high-cost mortgage,1.419479,1.419479,4.134967
162,initially_payable_to_institution_Not applicable,-0.771464,0.771464,0.462336
...,...,...,...,...
28,derived_race_Native Hawaiian or Other Pacific ...,-0.002997,0.002997,0.997007
136,co-applicant_sex_Information not provided by a...,-0.002187,0.002187,0.997815
98,co-applicant_credit_score_type_VantageScore 2.0,0.001712,0.001712,1.001713
72,debt_to_income_ratio_42,0.000330,0.000330,1.000330


In [109]:
coefficient_imp_2018 = filter_features_by_prefix(coefficient_df_2018)
coefficient_imp_2018

,Feature,Coefficient,abs_coefficient,odds_ratio
21,derived_ethnicity_Hispanic or Latino,0.209138,0.209138,1.232615
107,applicant_ethnicity_observed_Not applicable,-0.171393,0.171393,0.842490
125,applicant_race_observed_Not applicable,-0.148836,0.148836,0.861711
100,applicant_ethnicity-1_Hispanic or Latino,0.114214,0.114214,1.120992
103,co-applicant_ethnicity-1_Hispanic or Latino,0.075874,0.075874,1.078826
22,derived_ethnicity_Joint,-0.056099,0.056099,0.945446
29,derived_race_Race Not Available,-0.050488,0.050488,0.950765
115,applicant_race-1_Information not provided by a...,-0.048077,0.048077,0.953060
112,applicant_race-1_American Indian or Alaska Native,-0.040649,0.040649,0.960166
25,derived_race_Asian,-0.039442,0.039442,0.961326


## 2019

In [110]:
coefficient_sorted_2019 = deletezero_sort(coefficient_df_2019)
coefficient_sorted_2019

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_91328/2794036433.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['abs_coefficient'] = df['Coefficient'].abs()
/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_91328/2794036433.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['odds_ratio'] = np.exp(df['Coefficient'])


,Feature,Coefficient,abs_coefficient,odds_ratio
0,const,4.823885,4.823885,124.447669
39,purchaser_type_Not applicable,-2.201635,2.201635,0.110622
51,hoepa_status_Not applicable,-1.481915,1.481915,0.227202
50,hoepa_status_Not a high-cost mortgage,1.436190,1.436190,4.204646
158,initially_payable_to_institution_Not applicable,-0.829595,0.829595,0.436226
...,...,...,...,...
10,tract_owner_occupied_units,0.004563,0.004563,1.004574
94,co-applicant_credit_score_type_More than one c...,-0.004306,0.004306,0.995703
149,co-applicant_age_45-54,-0.001839,0.001839,0.998163
143,applicant_age_45-54,0.001447,0.001447,1.001448


In [111]:
coefficient_imp_2019 = filter_features_by_prefix(coefficient_df_2019)
coefficient_imp_2019

,Feature,Coefficient,abs_coefficient,odds_ratio
21,derived_ethnicity_Hispanic or Latino,0.209165,0.209165,1.232648
124,applicant_race_observed_Not applicable,-0.171518,0.171518,0.842385
118,co-applicant_race-1_Asian,0.169724,0.169724,1.184978
99,applicant_ethnicity-1_Hispanic or Latino,0.154446,0.154446,1.167011
106,applicant_ethnicity_observed_Not applicable,-0.151002,0.151002,0.859846
120,co-applicant_race-1_Information not provided b...,-0.100055,0.100055,0.904787
111,applicant_race-1_American Indian or Alaska Native,-0.064000,0.064000,0.938005
22,derived_ethnicity_Joint,0.058037,0.058037,1.059754
117,co-applicant_race-1_American Indian or Alaska ...,-0.054708,0.054708,0.946761
112,applicant_race-1_Asian,-0.053643,0.053643,0.947770


## 2020

In [112]:
coefficient_sorted_2020 = deletezero_sort(coefficient_df_2020)
coefficient_sorted_2020

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_91328/2794036433.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['abs_coefficient'] = df['Coefficient'].abs()
/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_91328/2794036433.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['odds_ratio'] = np.exp(df['Coefficient'])


,Feature,Coefficient,abs_coefficient,odds_ratio
0,const,4.910882e+00,4.910882e+00,135.759083
40,purchaser_type_Not applicable,-1.961072e+00,1.961072e+00,0.140707
52,hoepa_status_Not applicable,-1.623196e+00,1.623196e+00,0.197267
51,hoepa_status_Not a high-cost mortgage,1.457598e+00,1.457598e+00,4.295627
158,initially_payable_to_institution_Not applicable,-6.801942e-01,6.801942e-01,0.506519
...,...,...,...,...
28,derived_race_Native Hawaiian or Other Pacific ...,2.522188e-03,2.522188e-03,1.002525
106,co-applicant_ethnicity-1_Not applicable,1.694006e-03,1.694006e-03,1.001695
26,derived_race_Black or African American,-6.115440e-04,6.115440e-04,0.999389
70,debt_to_income_ratio_41,-7.295314e-06,7.295314e-06,0.999993


In [113]:
coefficient_imp_2020 = filter_features_by_prefix(coefficient_df_2020)
coefficient_imp_2020

,Feature,Coefficient,abs_coefficient,odds_ratio
124,applicant_race_observed_Not applicable,-3.927088e-01,3.927088e-01,0.675225
21,derived_ethnicity_Hispanic or Latino,2.620162e-01,2.620162e-01,1.299548
115,applicant_race-1_Information not provided by a...,-2.083749e-01,2.083749e-01,0.811903
29,derived_race_Race Not Available,-2.083385e-01,2.083385e-01,0.811932
107,applicant_ethnicity_observed_Not applicable,-1.773947e-01,1.773947e-01,0.837449
108,applicant_ethnicity_observed_Not collected on ...,-1.256098e-01,1.256098e-01,0.881959
103,co-applicant_ethnicity-1_Hispanic or Latino,1.069788e-01,1.069788e-01,1.112911
27,derived_race_Joint,-9.997374e-02,9.997374e-02,0.904861
101,applicant_ethnicity-1_Information not provided...,9.765991e-02,9.765991e-02,1.102588
20,derived_ethnicity_Ethnicity Not Available,9.722106e-02,9.722106e-02,1.102104


## 2021

In [114]:
coefficient_sorted_2021 = deletezero_sort(coefficient_df_2021)
coefficient_sorted_2021

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_91328/2794036433.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['abs_coefficient'] = df['Coefficient'].abs()
/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_91328/2794036433.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['odds_ratio'] = np.exp(df['Coefficient'])


,Feature,Coefficient,abs_coefficient,odds_ratio
0,const,4.075107,4.075107,58.856762
50,hoepa_status_Not applicable,-1.471332,1.471332,0.229619
39,purchaser_type_Not applicable,-1.432041,1.432041,0.238821
49,hoepa_status_Not a high-cost mortgage,1.335773,1.335773,3.802934
37,purchaser_type_Ginnie Mae,0.682367,0.682367,1.978555
...,...,...,...,...
82,applicant_credit_score_type_Experian Fair Isaac,-0.002445,0.002445,0.997558
13,conforming_loan_limit_NC,-0.001167,0.001167,0.998834
107,applicant_ethnicity_observed_Not collected on ...,-0.001135,0.001135,0.998866
134,co-applicant_sex_Information not provided by a...,0.000802,0.000802,1.000802


In [115]:
coefficient_imp_2021 = filter_features_by_prefix(coefficient_df_2021)
coefficient_imp_2021

,Feature,Coefficient,abs_coefficient,odds_ratio
22,derived_ethnicity_Joint,0.135085,0.135085,1.144634
127,co-applicant_race_observed_Not collected on th...,0.107237,0.107237,1.113198
100,applicant_ethnicity-1_Hispanic or Latino,-0.104972,0.104972,0.900350
126,co-applicant_race_observed_Not applicable,-0.100327,0.100327,0.904541
106,applicant_ethnicity_observed_Not applicable,-0.090431,0.090431,0.913537
123,applicant_race_observed_Not applicable,-0.089167,0.089167,0.914693
118,co-applicant_race-1_Asian,-0.083836,0.083836,0.919582
23,derived_race_2 or more minority races,0.068735,0.068735,1.071153
21,derived_ethnicity_Hispanic or Latino,-0.068065,0.068065,0.934200
25,derived_race_Asian,0.066700,0.066700,1.068975


## 2022

In [116]:
coefficient_sorted_2022 = deletezero_sort(coefficient_df_2022)
coefficient_sorted_2022

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_91328/2794036433.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['abs_coefficient'] = df['Coefficient'].abs()
/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_91328/2794036433.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['odds_ratio'] = np.exp(df['Coefficient'])


,Feature,Coefficient,abs_coefficient,odds_ratio
0,const,4.422911,4.422911,83.338502
50,hoepa_status_Not applicable,-1.883501,1.883501,0.152057
49,hoepa_status_Not a high-cost mortgage,1.610663,1.610663,5.006131
39,purchaser_type_Not applicable,-1.505383,1.505383,0.221932
33,"purchaser_type_Commercial bank, savings bank, ...",0.892676,0.892676,2.441655
...,...,...,...,...
116,applicant_race-1_Native Hawaiian or Other Paci...,-0.002399,0.002399,0.997604
92,co-applicant_credit_score_type_FICO Risk Score...,-0.002043,0.002043,0.997959
93,co-applicant_credit_score_type_FICO Risk Score...,-0.001473,0.001473,0.998528
124,co-applicant_race-1_Not applicable,-0.000395,0.000395,0.999605


In [117]:
coefficient_imp_2022 = filter_features_by_prefix(coefficient_df_2022)
coefficient_imp_2022

,Feature,Coefficient,abs_coefficient,odds_ratio
21,derived_ethnicity_Hispanic or Latino,-0.175504,0.175504,0.839034
100,applicant_ethnicity-1_Hispanic or Latino,-0.164934,0.164934,0.847950
25,derived_race_Asian,0.148164,0.148164,1.159703
113,applicant_race-1_Asian,0.141871,0.141871,1.152428
107,applicant_ethnicity_observed_Not applicable,-0.094287,0.094287,0.910021
125,applicant_race_observed_Not applicable,-0.092189,0.092189,0.911933
27,derived_race_Joint,-0.088956,0.088956,0.914885
101,applicant_ethnicity-1_Information not provided...,-0.080068,0.080068,0.923054
20,derived_ethnicity_Ethnicity Not Available,-0.080068,0.080068,0.923054
26,derived_race_Black or African American,-0.075621,0.075621,0.927168


In [118]:
len(coefficient_sorted_2018), len(coefficient_sorted_2019), len(coefficient_sorted_2020), len(coefficient_sorted_2021), len(coefficient_sorted_2022)

(80, 130, 119, 127, 162)